In [ ]:
from numpy import pi
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_bloch_multivector, array_to_latex

#### Source: 

`https://github.com/Qiskit/textbook/blob/main/notebooks/ch-algorithms/quantum-fourier-transform.ipynb`

The discrete Fourier transform acts on a vector $(x_0, ..., x_{N-1})$ and maps it to the vector $(y_0, ..., y_{N-1})$ according to the formula


$$y_k = \frac{1}{\sqrt{N}}\sum_{j=0}^{N-1}x_j\omega_N^{jk}$$


where $\omega_N^{jk} = e^{2\pi i \frac{jk}{N}}$.

Similarly, the quantum Fourier transform acts on a quantum state $\vert X\rangle = \sum_{j=0}^{N-1} x_j \vert j \rangle$ and maps it to the quantum state $\vert Y\rangle = \sum_{k=0}^{N-1} y_k \vert k \rangle$ according to the formula


$$y_k = \frac{1}{\sqrt{N}}\sum_{j=0}^{N-1}x_j\omega_N^{jk}$$


with $\omega_N^{jk}$ defined as above. Note that only the amplitudes of the state were affected by this transformation.

This can also be expressed as the map:


$$\vert j \rangle \mapsto \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}\omega_N^{jk} \vert k \rangle$$



Or the unitary matrix:


$$ U_{QFT} = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1} \sum_{k=0}^{N-1} \omega_N^{jk} \vert k \rangle \langle j \vert$$

#### QFT on 3 qubits

In [ ]:
qc = QuantumCircuit(3)
qc.h(2)
qc.draw()

In [ ]:
qc.cp(pi/2, 1, 2) # CROT from qubit 1 to qubit 2
qc.draw()

In [ ]:
qc.cp(pi/4, 0, 2) # CROT from qubit 2 to qubit 0
qc.draw()

In [ ]:
qc.h(1)
qc.cp(pi/2, 0, 1) # CROT from qubit 0 to qubit 1
qc.h(0)
qc.draw()

In [ ]:
qc.swap(0,2)
qc.draw()

#### General QFT

$\bullet~$ a routine to implement the controlled rotations

In [ ]:
def qft_rotations(circuit, n):
    if n == 0:
        return circuit
    n -= 1
    circuit.h(n)
    for qubit in range(n):
        circuit.cp(pi/2**(n-qubit), qubit, n)
    qft_rotations(circuit, n)

In [ ]:
qc = QuantumCircuit(3)
qft_rotations(qc,3)
qc.draw()

$\bullet~$ a routine to swap the readout register and do the full QFT

In [ ]:
def swap_registers(circuit, n):
    for qubit in range(n//2):
        circuit.swap(qubit, n-qubit-1)
    return circuit

def qft(circuit, n):
    """QFT on the first n qubits in circuit"""
    qft_rotations(circuit, n)
    swap_registers(circuit, n)
    return circuit

$\bullet~$ simulate the transformation of the statevector through the circuit

In [ ]:
backend = Aer.get_backend('statevector_simulator')

$\bullet~$ QFT on 4 qubits

In [ ]:
qc = QuantumCircuit(4)

$\bullet~$ initial statevector of the qubits: binary representation in the z-basis of the numbers  $~0, 1, \ldots , 15$

In [ ]:
qc.x(0)
qc.x(1)

$\bullet~$ run a simulation to get the initial statevector

In [ ]:
job = backend.run(transpile(qc, backend))
sv = job.result().get_statevector()
array_to_latex(sv, prefix="|\\psi_{f}\\rangle =")

In [ ]:
sv.draw('bloch')

$\bullet~$ complete the circuit to do the QFT

In [ ]:
qft(qc,4)
#qc.draw()

$\bullet~$ run a simulation to get the Fourier transform

In [ ]:
job = backend.run(transpile(qc, backend))
sv = job.result().get_statevector()
array_to_latex(sv, prefix="|\\psi_{f}\\rangle =")

In [ ]:
sv.draw('bloch')

#### Animated gif's from: 

`https://github.com/Qiskit/textbook/blob/main/notebooks/ch-algorithms/quantum-fourier-transform.ipynb`

#### Counting in the z-basis

![zbasiscounting](images/zbasis-counting.gif)

#### Counting in the Fourier basis

![fbasiscounting](images/fourierbasis-counting.gif)